In [53]:
# Importing libraries
#matplotlib inline
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



import re
from textwrap import wrap
from collections import Counter
from string import punctuation
from nltk import word_tokenize, ngrams

# Model evaluation and processing libraries
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, f1_score


In [54]:
df = pd.read_csv("/Users/zahrita/Documents/DissData/Output/corpusp1/HandcodOLTUPDATED.csv", encoding = "latin1")

In [77]:
df2 = pd.read_csv("/Users/zahrita/Documents/DissData/Output/corpusp1/HandcodOLT3CA1.csv", encoding = "latin1")

In [78]:
df3 = pd.read_csv("/Users/zahrita/Documents/DissData/Code/Process/QTAhand1.csv", encoding = "latin1")

In [79]:
df3 = df3.dropna()

In [80]:
df3.tail()

,date,volume,number,sentences,count,id,likely_pos,code
145,8/1/08,9.0,31.0,with our position so often emphasized and so g...,47.0,s2126,True,0.0
146,3/10/06,7.0,7.0,the japanese and korean exclusion league met a...,22.0,s1840,True,4.0
147,3/14/14,15.0,11.0,you can sleep sound also and not be worried wi...,58.0,s3440,True,2.0
148,1/3/03,3.0,49.0,"any toiler who, possessed of the knowledge tha...",88.0,s766,True,2.0
149,12/23/11,12.0,51.0,"craig's tireless endeavors, many school boards...",52.0,s2928,True,2.0


In [81]:
df1 = pd.concat([df, df2, df3], ignore_index = True)

In [82]:
df1.tail()

,date,volume,number,sentences,count,id,likely_pos,sample_type,code
946,8/1/08,9.0,31.0,with our position so often emphasized and so g...,47.0,s2126,True,NaN,0.0
947,3/10/06,7.0,7.0,the japanese and korean exclusion league met a...,22.0,s1840,True,NaN,4.0
948,3/14/14,15.0,11.0,you can sleep sound also and not be worried wi...,58.0,s3440,True,NaN,2.0
949,1/3/03,3.0,49.0,"any toiler who, possessed of the knowledge tha...",88.0,s766,True,NaN,2.0
950,12/23/11,12.0,51.0,"craig's tireless endeavors, many school boards...",52.0,s2928,True,NaN,2.0


In [83]:
df1.shape

(951, 9)

In [84]:
df1['sentences'] = df1['sentences'].str.replace(r"[!.\-,]+", "", regex=True)

In [85]:
df1['sentences'] = df1['sentences'].str.replace(r"\s+", " ", regex=True)

In [86]:
df1['code'] = df1['code'].replace(5, 3) #collapsing code 5 into three
df1['code'] = df1['code'].replace(3, 2) #collapsing code 3 into two for split
df1['code'] = df1['code'].replace(4, 3)

In [87]:
X = df1["sentences"]
y = df1["code"]

In [88]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42, stratify=y)

In [98]:
from imblearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
from imblearn.over_sampling import SMOTE
model = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words="english", min_df = 2, 
                              max_df=0.80, max_features=1000, norm = 'l2', )),
    ('lr', LogisticRegression(C=0.5, class_weight='balanced', 
                              multi_class='multinomial', solver='lbfgs',))
])
model.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.8, max_features=1000, min_df=2,
                                 stop_words='english')),
                ('lr',
                 LogisticRegression(C=0.5, class_weight='balanced',
                                    multi_class='multinomial'))])

In [99]:
y_pred = model.predict(X_train)
y_tp = model.predict(X_test)

print("Number of mislabeled points in the traing set out of a total %d points: %d" % (X_train.shape[0], (y_train != y_pred).sum()))
print("Number of mislabeled points in the testing set out of a total %d points: %d" % (X_test.shape[0], (y_test != y_tp).sum()))

Number of mislabeled points in the traing set out of a total 713 points: 40
Number of mislabeled points in the testing set out of a total 238 points: 36


In [100]:
train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)
print('Score', train_score, test_score)

Score 0.9438990182328191 0.8487394957983193


In [101]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_tp))

              precision    recall  f1-score   support

         0.0       0.94      0.91      0.93       173
         1.0       0.61      0.73      0.67        15
         2.0       0.52      0.55      0.53        29
         3.0       0.81      0.81      0.81        21

    accuracy                           0.85       238
   macro avg       0.72      0.75      0.73       238
weighted avg       0.86      0.85      0.85       238



In [103]:
df3 = pd.read_csv('/Users/zahrita/Documents/DissData/Data/Inputs/article_olv7.csv')

In [43]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify=y)

In [44]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.naive_bayes import ComplementNB
from imblearn.over_sampling import SMOTE

# Create a document-term matrix
vectorizer = CountVectorizer()
X1_train = vectorizer.fit_transform(X_train)
X2_test = vectorizer.transform(X_test)

sm = SMOTE(random_state=42, k_neighbors=2)
X_train_res, y_train_res = sm.fit_resample(X1_train, y_train)
nb = ComplementNB(fit_prior=False)
nb.fit(X_train_res, y_train_res)

ComplementNB(fit_prior=False)

In [45]:
y_pred = nb.predict(X_train_res)
y_tp = nb.predict(X2_test)
print("Number of mislabeled points in the traing set out of a total %d points: %d" % (X_train_res.shape[0], (y_train_res != y_pred).sum()))
print("Number of mislabeled points in the testing set out of a total %d points: %d" % (X2_test.shape[0], (y_test != y_tp).sum()))

Number of mislabeled points in the traing set out of a total 2208 points: 365
Number of mislabeled points in the testing set out of a total 191 points: 45


In [46]:
train_score = nb.score(X_train_res, y_train_res)
test_score = nb.score(X2_test, y_test)
print('Score', train_score, test_score)

Score 0.8346920289855072 0.7643979057591623


In [47]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_tp))

              precision    recall  f1-score   support

         0.0       0.89      0.83      0.86       139
         1.0       0.39      0.75      0.51        12
         2.0       0.53      0.39      0.45        23
         3.0       0.60      0.71      0.65        17

    accuracy                           0.76       191
   macro avg       0.60      0.67      0.62       191
weighted avg       0.79      0.76      0.77       191



In [104]:
df3['predicted_class'] = model.predict(df3['articles'])

In [105]:
df3.to_csv('qtafinaldf3.csv', index=False)